# Step 4: Evaluation with Updated Labels

Tests model annotation against the updated `final_human_label` column
in `step3_testset_500_human_completed_translated.csv`.

Model input: `text_block_en` (German text)
Ground truth: `final_human_label` (your updated labels)

## Setup

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import requests
import time
import os
import krippendorff
import importlib
from dotenv import load_dotenv
from sklearn.metrics import (
    accuracy_score, classification_report,
    f1_score, cohen_kappa_score, confusion_matrix
)
import annotation_setup
importlib.reload(annotation_setup)
from annotation_setup import VALID_LABELS, instructions, reminder, parse_label, parse_explanation

load_dotenv()
api_key = os.getenv("CAC_API_KEY")

HOST  = "https://maki.uni-mannheim.de/v1"
MODEL = "ministral-3-14b"

RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)

print(f"API key loaded: {'YES' if api_key else 'NO'}")
print(f"Model: {MODEL}")

API key loaded: YES
Model: ministral-3-14b


## Few-Shot Examples

In [2]:
FEW_SHOT_EXAMPLES = [
    # 1. Clear NO_CRIME_FRAME, integration context, no security link
    {
        "text": "Die Lage in der Unterkunft hat sich nach Angaben der Sozialarbeiter entspannt. Viele [Gruppe 1] haetten inzwischen Deutschkurse begonnen und bemuehten sich aktiv um eine Arbeitsstelle. Die Caritas lobte das Engagement der Freiwilligen, die regelmaessig Unterstuetzung anbieten.",
        "label": "NO_CRIME_FRAME",
        "explanation": "Der Absatz beschreibt Integration und ehrenamtliche Hilfe ohne jeden Bezug zu Kriminalitaet, Sicherheit oder Praevention."
    },
    # 2. Clear CRIME_FRAME_NEG, direct perpetrator
    {
        "text": "Ein 34-jaehriger [Gruppe 1] wurde am Freitagabend am Hauptbahnhof festgenommen. Er soll in den vergangenen Wochen mehrfach Passanten mit einem Messer bedroht und Handtaschen entrissen haben. Die Polizei hat ihn als Serientaeter eingestuft.",
        "label": "CRIME_FRAME_NEG",
        "explanation": "Ein Gruppenmitglied wird explizit als Taeter schwerer Straftaten dargestellt und direkt mit Kriminalitaet verknuepft."
    },
    # 3. CRIME_FRAME_NEG, institutional security framing (fixes missed cases)
    {
        "text": "Die Innenministerkonferenz hat beschlossen, die erkennungsdienstliche Erfassung aller einreisenden [Gruppe 1] zu verschaerfen. Innenminister Hoffmann erklaerte, man koenne sich keine Sicherheitsluecken leisten. Es muesse sichergestellt werden, dass keine Schleuser, Kriminellen oder Extremisten die Situation ausnutzten.",
        "label": "CRIME_FRAME_NEG",
        "explanation": "Auch ohne konkrete Straftat wird die Gruppe institutionell als potenzielle Sicherheitsgefahr gerahmt — die explizite Verknuepfung mit Kriminalitaet und Extremismus durch eine Regierungsstimme ist CRIME_FRAME_NEG."
    },
    # 4. NO_CRIME_FRAME, group is victim of violence
    {
        "text": "Auf dem Gelaende der Fluechtlingsunterkunft in Tempelhof wurden in der Nacht Fensterscheiben eingeworfen und Hakenkreuze gesprueht. Die Polizei ermittelt wegen schwerer Sachbeschaedigung und Volksverhetzung. Die [Gruppe 1] Bewohner wurden unverletzt, aber stark veraengstigt in eine Notunterkunft gebracht.",
        "label": "NO_CRIME_FRAME",
        "explanation": "Die [Gruppe 1] sind Opfer eines rechtsextremen Angriffs — eine Gruppe als Opfer von Gewalt darzustellen ist kein Sicherheitsrahmen gegen diese Gruppe."
    },
    # location not mentioned and no Germany link = NO_CRIME_FRAME
    {
        "text": "Berichte zufolge wurden mehrere [Gruppe 1] bei Auseinandersetzungen verletzt. Wo genau die Vorfaelle stattfanden, blieb unklar. Ein Bezug zu Deutschland oder deutschen Sicherheitsbehoerden wurde nicht hergestellt.",
        "label": "NO_CRIME_FRAME",
        "explanation": "Kein expliziter Bezug zu Deutschland oder zur deutschen inneren Sicherheit — ohne diesen Bezug gilt NO_CRIME_FRAME."
    },
    # 5. NO_CRIME_FRAME, international war reporting between states
    {
        "text": "Die [Gruppe 1] Streitkraefte haben nach Angaben des Verteidigungsministeriums in Kiew erneut mehrere Staedte im Osten beschossen. Bei den Angriffen kamen nach ukrainischen Angaben mindestens zwoelf Zivilisten ums Leben. Die NATO beriet ueber weitere Waffenlieferungen.",
        "label": "NO_CRIME_FRAME",
        "explanation": "Kriegsberichterstattung ueber einen Konflikt zwischen Staaten ohne Bezug zu Minderheiten in Deutschland erhaelt immer NO_CRIME_FRAME."
    },
    # Germany-link exception, foreign crime but explicit Germany connection = CRIME_FRAME_NEG
    {
        "text": "Der Attentaeter, der in Wien drei Menschen toetete, hatte mehrere Jahre in Deutschland gelebt und war deutschen Behoerden als Gefaehrder bekannt. Das Bundesinnenministerium prueft nun, ob Sicherheitsluecken im deutschen System vorlagen.",
        "label": "CRIME_FRAME_NEG",
        "explanation": "Obwohl die Tat in Oesterreich stattfand, wird ein expliziter Bezug zur deutschen Sicherheit hergestellt — der Taeter lebte in Deutschland und deutsche Behoerden sind beteiligt. Das ergibt CRIME_FRAME_NEG."
    },
    # 6. CRIME_FRAME_NEG, group legitimizes terrorism without being direct perpetrator
    {
        "text": "Sprecher der [Gruppe 1] bezeichneten den Anschlag als legitimen Widerstand und erklaerten, die Opfer seien selbst schuld. In sozialen Medien verbreiteten Anhaenger der Gruppe Videos, in denen der Terrorakt gefeiert wurde.",
        "label": "CRIME_FRAME_NEG",
        "explanation": "Die Gruppe legitimiert und feiert explizit einen Terroranschlag — das zaehlt als CRIME_FRAME_NEG auch ohne direkte Taeterschaft."
    },
    # 7. CRIME_FRAME_POS, successful security operation
    {
        "text": "Das Gemeinsame Terrorismusabwehrzentrum hat nach intensiver Zusammenarbeit von Polizei und Geheimdiensten einen islamistischen Anschlag in Muenchen vereitelt. Drei Verdaechtige wurden festgenommen. Innenministerin Weber sprach von einem grossen Erfolg der Sicherheitsbehoerden.",
        "label": "CRIME_FRAME_POS",
        "explanation": "Die Sicherheitsbehoerden werden explizit als erfolgreiche Akteure gerahmt die eine Bedrohung abgewendet haben — Schutz und Sicherheitsgewinn stehen im Vordergrund."
    },
    # 8. CRIME_FRAME_POS, social factors explanation + differentiation (aligns with Robin's definition)
    {
        "text": "Experten erklaeren den Anstieg der Jugendkriminalitaet in Brennpunktvierteln vor allem mit sozialer Ausgrenzung und fehlenden Perspektiven fuer [Gruppe 1] Jugendliche. Kriminologin Mueller betonte, man duerfe keinen Generalverdacht gegen die Gruppe hegen — die grosse Mehrheit halte sich an Gesetze.",
        "label": "CRIME_FRAME_POS",
        "explanation": "Kriminalitaet wird auf soziale Ursachen zurueckgefuehrt und eine explizite Differenzierung wird vorgenommen — das ist CRIME_FRAME_POS nach der Praventions- und Differenzierungsdefinition."
    },
    # 9. CRIME_FRAME_POS, prevention/deradicalization program
    {
        "text": "Das neue Deradikalisierungsprogramm der Bundesregierung soll vor allem [Gruppe 1] Jugendliche ansprechen, die gefaehrdet sind, in extremistische Kreise abzudriften. Sozialarbeiter begleiten die Teilnehmer ueber zwei Jahre und helfen beim Einstieg in Ausbildung und Arbeit.",
        "label": "CRIME_FRAME_POS",
        "explanation": "Ein explizites Praeventionsprogramm im Sicherheitskontext — Schutz und Praevention stehen im Vordergrund, nicht Taeterschaft."
    },
    # 10. NO_CRIME_FRAME, integration without security frame (boundary with CRIME_FRAME_POS)
    {
        "text": "Die Stadtratsfraktion diskutierte Massnahmen zur besseren Integration von [Gruppe 1]. Themen waren Sprachkurse, Wohnraumvermittlung und Beschaeftigungsfoerderung. Ein konkreter Sicherheits- oder Kriminalitaetsbezug wurde nicht hergestellt.",
        "label": "NO_CRIME_FRAME",
        "explanation": "Integration ohne jeden Sicherheits- oder Kriminalitaetsbezug ist NO_CRIME_FRAME — CRIME_FRAME_POS erfordert einen expliziten Sicherheits- oder Praeventionsrahmen."
    },
    # fixes id 14, 21, 135, generalised threat/disorder framing
    {
        "text": "Die Behörden berichten, dass sich in Teilen der Stadt regelrechte Parallelgesellschaften gebildet haben, in denen [Gruppe 1] unter sich bleiben und staatliche Kontrolle kaum noch greift. Anwohner klagen über Verwahrlosung und mangelnde Sicherheit.",
        "label": "CRIME_FRAME_NEG",
        "explanation": "Die Gruppe wird mit Kontrollverlust, Unordnung und mangelnder Sicherheit im deutschen Inland verknüpft — das ist generalisierte gesellschaftliche Bedrohungsrahmung und zählt als CRIME_FRAME_NEG."
    },

    # fixes id 30, explicit domestic security risk framing
    {
        "text": "Der Botschafter warnte: 'Diese [Gruppe 1] stellen ein Sicherheitsrisiko für Deutschland dar.' Es sei falsch, sie aufzunehmen, da sie in deutschen Städten für Unruhe sorgen würden.",
        "label": "CRIME_FRAME_NEG",
        "explanation": "Die Gruppe wird von einer öffentlichen Person explizit als Sicherheitsrisiko für Deutschland bezeichnet — das ist CRIME_FRAME_NEG unabhängig davon ob ein konkretes Verbrechen genannt wird."
    },
    # fixes id 15, 47, 113, 123, 189, political commentary overcalling
    {
        "text": "Die Politikerin forderte eine bessere Integrationspolitik für [Gruppe 1]. In der Debatte wurden auch Fragen der öffentlichen Ordnung angesprochen, ohne jedoch konkrete Vorfälle oder Straftaten zu nennen.",
        "label": "NO_CRIME_FRAME",
        "explanation": "Allgemeine politische Debatten über Integration oder öffentliche Ordnung ohne explizite Verknüpfung der Gruppe mit konkreten Straftaten, Verdacht oder Ermittlungen sind NO_CRIME_FRAME."
    }
]

print(f"Few-shot examples loaded: {len(FEW_SHOT_EXAMPLES)}")
for i, ex in enumerate(FEW_SHOT_EXAMPLES):
    print(f"  {i+1}. {ex['label']}")

Few-shot examples loaded: 15
  1. NO_CRIME_FRAME
  2. CRIME_FRAME_NEG
  3. CRIME_FRAME_NEG
  4. NO_CRIME_FRAME
  5. NO_CRIME_FRAME
  6. NO_CRIME_FRAME
  7. CRIME_FRAME_NEG
  8. CRIME_FRAME_NEG
  9. CRIME_FRAME_POS
  10. CRIME_FRAME_POS
  11. CRIME_FRAME_POS
  12. NO_CRIME_FRAME
  13. CRIME_FRAME_NEG
  14. CRIME_FRAME_NEG
  15. NO_CRIME_FRAME


## Core Functions

In [3]:
def call_api(messages, temperature=0.0):
    payload = {
        "model": MODEL,
        "temperature": temperature,
        "messages": messages
    }
    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json"
    }
    for attempt in range(3):
        try:
            r = requests.post(
                f"{HOST}/chat/completions",
                json=payload, headers=headers, timeout=120
            )
            if r.status_code == 200:
                return r.json()["choices"][0]["message"]["content"].strip()
            print(f"  [attempt {attempt+1}] status {r.status_code}")
        except Exception as e:
            print(f"  [attempt {attempt+1}] error: {e}")
            time.sleep(5)
    return "ERROR"


def format_few_shot_block(examples):
    blocks = []
    for ex in examples:
        block = f"Text:\n{ex['text']}\n\nLabel: {ex['label']}\nExplanation: {ex['explanation']}"
        blocks.append(block)
    return "\n\n---\n\n".join(blocks)


def annotate_few_shot(text, temperature=0.0):
    few_shot_block = format_few_shot_block(FEW_SHOT_EXAMPLES)
    prompt = (
        f"{instructions}\n\n"
        f"Hier sind Beispiele fuer die Annotation:\n\n{few_shot_block}\n\n---\n\n"
        f"Jetzt annotieren Sie diesen Absatz:\n\n{text}\n\n"
        f"{reminder}\n\n"
        "Antworten Sie genau in diesem Format:\n"
        "Label: <NO_CRIME_FRAME oder CRIME_FRAME_NEG oder CRIME_FRAME_POS>\n"
        "Explanation: <ein Satz>"
    )
    messages = [
        {"role": "system", "content": "Sie sind ein Experte fuer Inhaltsanalyse. Antworten Sie immer im angegebenen Format."},
        {"role": "user",   "content": prompt}
    ]
    return call_api(messages, temperature)


def parse_reasoning(raw):
    for line in raw.split("\n"):
        if line.lower().startswith("reasoning:"):
            return line[len("reasoning:"):].strip()
    return ""


def annotate_few_shot_cot(text, temperature=0.0):
    few_shot_block = format_few_shot_block(FEW_SHOT_EXAMPLES)
    prompt = (
        f"{instructions}\n\n"
        f"Hier sind Beispiele mit Begruendungen:\n\n{few_shot_block}\n\n---\n\n"
        f"Beantworten Sie vor der Annotation diese vier Fragen:\n"
        f"1. Ist die genannte Gruppe Taeter oder Opfer?\n"
        f"2. Findet das Ereignis in Deutschland statt oder gibt es einen expliziten Bezug zur deutschen Sicherheit?\n"
        f"3. Wird eine Sicherheitsmassnahme als expliziter Erfolg gerahmt?\n"
        f"4. Wird die Gruppe explizit als Quelle von Bedrohung oder Kriminalitaet in Deutschland dargestellt?\n\n"
        f"Text:\n\n{text}\n\n"
        f"{reminder}\n\n"
        "Antworten Sie genau in diesem Format:\n"
        "Reasoning: <kurze Antworten auf die vier Fragen>\n"
        "Label: <NO_CRIME_FRAME oder CRIME_FRAME_NEG oder CRIME_FRAME_POS>\n"
        "Explanation: <ein Satz>"
    )
    messages = [
        {"role": "system", "content": "Sie sind ein Experte fuer Inhaltsanalyse. Beantworten Sie zuerst die Kontrollfragen, dann vergeben Sie das Label."},
        {"role": "user",   "content": prompt}
    ]
    return call_api(messages, temperature)


print("Functions loaded.")

Functions loaded.


## Load Data

In [4]:
human_completed = pd.read_csv(
    RESULTS_DIR / "step3_testset_500_human_completed_translated.csv"
)

human_completed = human_completed[
    human_completed["final_human_label"].isin(VALID_LABELS)
].copy().reset_index(drop=True)

print(f"Rows: {len(human_completed)}")
print("Ground truth distribution:")
print(human_completed["final_human_label"].value_counts())

Rows: 250
Ground truth distribution:
final_human_label
NO_CRIME_FRAME     131
CRIME_FRAME_NEG    113
CRIME_FRAME_POS      6
Name: count, dtype: int64


## Condition B: Few-Shot Annotation

In [ ]:
OUTPUT_B = RESULTS_DIR / "step3_updated_fewshot.csv"

if OUTPUT_B.exists():
    done_b   = pd.read_csv(OUTPUT_B)
    done_ids = set(done_b["testset_id"])
    print(f"Resuming: {len(done_b)} done, {len(human_completed) - len(done_b)} remaining")
else:
    done_b   = pd.DataFrame()
    done_ids = set()
    print(f"Starting fresh: {len(human_completed)} rows")

buffer = []

for i, row in human_completed.iterrows():
    if row["testset_id"] in done_ids:
        continue

    raw         = annotate_few_shot(str(row["text_block_en"]))
    label       = parse_label(raw)
    explanation = parse_explanation(raw)

    buffer.append({
        "testset_id":          row["testset_id"],
        "group":               row["group"],
        "text_block_en":       row["text_block_en"],
        "final_human_label":   row["final_human_label"],
        "fewshot_label":       label,
        "fewshot_explanation": explanation,
    })
    done_ids.add(row["testset_id"])

    if len(buffer) % 50 == 0:
        chunk  = pd.DataFrame(buffer)
        chunk.to_csv(OUTPUT_B, mode="a", header=not OUTPUT_B.exists(), index=False)
        done_b = pd.concat([done_b, chunk], ignore_index=True)
        buffer = []
        print(f"  [{len(done_b)}/{len(human_completed)}] checkpointed")

if buffer:
    chunk  = pd.DataFrame(buffer)
    chunk.to_csv(OUTPUT_B, mode="a", header=not OUTPUT_B.exists(), index=False)
    done_b = pd.concat([done_b, chunk], ignore_index=True)

print(f"\nCondition B done. {len(done_b)} rows.")
print(done_b["fewshot_label"].value_counts())

Starting fresh: 250 rows
  [50/250] checkpointed
  [100/250] checkpointed
  [150/250] checkpointed
  [200/250] checkpointed
  [250/250] checkpointed

Condition B done. 250 rows.
fewshot_label
NO_CRIME_FRAME     154
CRIME_FRAME_NEG     95
CRIME_FRAME_POS      1
Name: count, dtype: int64


## Condition C: Few-Shot + CoT

In [ ]:
OUTPUT_C = RESULTS_DIR / "step3_updated_fewshot_cot.csv"

if OUTPUT_C.exists():
    done_c   = pd.read_csv(OUTPUT_C)
    done_ids = set(done_c["testset_id"])
    print(f"Resuming: {len(done_c)} done, {len(human_completed) - len(done_c)} remaining")
else:
    done_c   = pd.DataFrame()
    done_ids = set()
    print(f"Starting fresh: {len(human_completed)} rows")

buffer = []

for i, row in human_completed.iterrows():
    if row["testset_id"] in done_ids:
        continue

    raw         = annotate_few_shot_cot(str(row["text_block_en"]))
    label       = parse_label(raw)
    explanation = parse_explanation(raw)
    reasoning   = parse_reasoning(raw)

    buffer.append({
        "testset_id":        row["testset_id"],
        "group":             row["group"],
        "text_block_en":     row["text_block_en"],
        "final_human_label": row["final_human_label"],
        "cot_label":         label,
        "cot_reasoning":     reasoning,
        "cot_explanation":   explanation,
    })
    done_ids.add(row["testset_id"])

    if len(buffer) % 50 == 0:
        chunk  = pd.DataFrame(buffer)
        chunk.to_csv(OUTPUT_C, mode="a", header=not OUTPUT_C.exists(), index=False)
        done_c = pd.concat([done_c, chunk], ignore_index=True)
        buffer = []
        print(f"  [{len(done_c)}/{len(human_completed)}] checkpointed")

if buffer:
    chunk  = pd.DataFrame(buffer)
    chunk.to_csv(OUTPUT_C, mode="a", header=not OUTPUT_C.exists(), index=False)
    done_c = pd.concat([done_c, chunk], ignore_index=True)

print(f"\nCondition C done. {len(done_c)} rows.")
print(done_c["cot_label"].value_counts())

Starting fresh: 250 rows
  [50/250] checkpointed
  [100/250] checkpointed
  [150/250] checkpointed
  [200/250] checkpointed
  [250/250] checkpointed

Condition C done. 250 rows.
cot_label
NO_CRIME_FRAME     151
CRIME_FRAME_NEG     97
CRIME_FRAME_POS      2
Name: count, dtype: int64


## Evaluation

In [7]:
def evaluate(y_true, y_pred, name):
    valid = [
        (t, p) for t, p in zip(y_true, y_pred)
        if t in VALID_LABELS and p in VALID_LABELS
    ]
    if not valid:
        print(f"{name}: no valid predictions")
        return {}
    yt, yp   = zip(*valid)
    acc      = accuracy_score(yt, yp)
    f1_macro = f1_score(yt, yp, average="macro", labels=VALID_LABELS, zero_division=0)
    f1_w     = f1_score(yt, yp, average="weighted", labels=VALID_LABELS, zero_division=0)
    kappa    = cohen_kappa_score(yt, yp, labels=VALID_LABELS)
    alpha_data = np.array([
        [VALID_LABELS.index(t) for t in yt],
        [VALID_LABELS.index(p) for p in yp]
    ])
    try:
        alpha = krippendorff.alpha(alpha_data, level_of_measurement="nominal")
    except Exception:
        alpha = float("nan")
    print(f"\n{'='*50}\nCondition: {name}\n{'='*50}")
    print(f"Rows: {len(yt)} | Accuracy: {acc:.3f} | Macro F1: {f1_macro:.3f} | Weighted F1: {f1_w:.3f}")
    print(f"Cohen's kappa: {kappa:.3f} | Krippendorff's alpha: {alpha:.3f}")
    print()
    print(classification_report(yt, yp, labels=VALID_LABELS, zero_division=0))
    print("Confusion matrix (rows=human, cols=model):")
    print(confusion_matrix(yt, yp, labels=VALID_LABELS))
    print(f"Label order: {VALID_LABELS}")
    return {"name": name, "n": len(yt), "accuracy": acc,
            "macro_f1": f1_macro, "weighted_f1": f1_w,
            "kappa": kappa, "alpha": alpha}


print("Evaluate function loaded.")

Evaluate function loaded.


In [8]:
done_b = pd.read_csv(OUTPUT_B)
res_b  = evaluate(
    done_b["final_human_label"].tolist(),
    done_b["fewshot_label"].tolist(),
    "B: Few-shot"
)


Condition: B: Few-shot
Rows: 250 | Accuracy: 0.792 | Macro F1: 0.624 | Weighted F1: 0.784
Cohen's kappa: 0.588 | Krippendorff's alpha: 0.586

                 precision    recall  f1-score   support

 NO_CRIME_FRAME       0.77      0.90      0.83       131
CRIME_FRAME_NEG       0.83      0.70      0.76       113
CRIME_FRAME_POS       1.00      0.17      0.29         6

       accuracy                           0.79       250
      macro avg       0.87      0.59      0.62       250
   weighted avg       0.80      0.79      0.78       250

Confusion matrix (rows=human, cols=model):
[[118  13   0]
 [ 34  79   0]
 [  2   3   1]]
Label order: ['NO_CRIME_FRAME', 'CRIME_FRAME_NEG', 'CRIME_FRAME_POS']


In [9]:
done_c = pd.read_csv(OUTPUT_C)
res_c  = evaluate(
    done_c["final_human_label"].tolist(),
    done_c["cot_label"].tolist(),
    "C: Few-shot + CoT"
)


Condition: C: Few-shot + CoT
Rows: 250 | Accuracy: 0.732 | Macro F1: 0.654 | Weighted F1: 0.728
Cohen's kappa: 0.472 | Krippendorff's alpha: 0.471

                 precision    recall  f1-score   support

 NO_CRIME_FRAME       0.72      0.82      0.77       131
CRIME_FRAME_NEG       0.75      0.65      0.70       113
CRIME_FRAME_POS       1.00      0.33      0.50         6

       accuracy                           0.73       250
      macro avg       0.82      0.60      0.65       250
   weighted avg       0.74      0.73      0.73       250

Confusion matrix (rows=human, cols=model):
[[108  23   0]
 [ 40  73   0]
 [  3   1   2]]
Label order: ['NO_CRIME_FRAME', 'CRIME_FRAME_NEG', 'CRIME_FRAME_POS']


In [ ]:
summary = pd.DataFrame([r for r in [res_b, res_c] if r])
summary = summary.set_index("name").round(3)
print("\n=== Summary ===")
print(summary[["n", "accuracy", "macro_f1", "weighted_f1", "kappa", "alpha"]].to_string())
summary.to_csv(RESULTS_DIR / "step3_updated_evaluation_summary.csv")


=== Summary ===
                     n  accuracy  macro_f1  weighted_f1  kappa  alpha
name                                                                 
B: Few-shot        250     0.792     0.624        0.784  0.588  0.586
C: Few-shot + CoT  250     0.732     0.654        0.728  0.472  0.471


## Error Analysis

In [11]:
# use whichever condition performed better
best     = done_b.copy()
best_col = "fewshot_label"

best["correct"] = best["final_human_label"] == best[best_col]
errors = best[~best["correct"]].copy()

print(f"Remaining errors: {len(errors)} / {len(best)}")
print()
print(errors.groupby(["final_human_label", best_col]).size().reset_index(name="count").to_string(index=False))

Remaining errors: 52 / 250

final_human_label   fewshot_label  count
  CRIME_FRAME_NEG  NO_CRIME_FRAME     34
  CRIME_FRAME_POS CRIME_FRAME_NEG      3
  CRIME_FRAME_POS  NO_CRIME_FRAME      2
   NO_CRIME_FRAME CRIME_FRAME_NEG     13


In [12]:
for (human_lbl, model_lbl), grp in errors.groupby(["final_human_label", best_col]):
    print(f"\n{'='*60}")
    print(f"Human: {human_lbl}  |  Model: {model_lbl}  |  Count: {len(grp)}")
    print(f"{'='*60}")
    for _, row in grp.head(5).iterrows():
        print(f"\nid:{row['testset_id']} | group:{row['group']}")
        print(f"Text: {str(row['text_block_en'])[:400]}")
        print(f"Explanation: {row['fewshot_explanation']}")
        print()


Human: CRIME_FRAME_NEG  |  Model: NO_CRIME_FRAME  |  Count: 34

id:14 | group:nan
Text: Die Einwohner des Stadtteils Chloraka monieren, dort habe sich eine Art Getto gebildet, in dem Hunderte [Gruppe 1] und [Gruppe 1] lebten und die Polizei die Kontrolle verloren habe, berichteten zyprische Medien übereinstimmend.

Die EU-Inselrepublik Zypern bittet immer wieder die anderen EU-Mitgliedstaaten darum, auf der Insel angekommene Schutzsuchende aufzunehmen. Allein im vergangenen Oktober u
Explanation: Der Absatz beschreibt zwar eine als problematisch wahrgenommene soziale Situation (Ghetto-Bildung) und den Verlust von Polizeikontrolle, jedoch **ohne explizite Verknüpfung mit Kriminalität, Gewalt, Sicherheitsbedrohung oder konkreten Straftaten** der Gruppe – zudem findet die Handlung **ausschließlich auf Zypern** statt, ohne Bezug zur deutschen inneren Sicherheit.


id:19 | group:ISLMST
Text: In Ägypten sind drei Journalisten nach rund eineinhalb Jahren Haft freigelassen worden. Diaa Raschw